# Python drills 9: data engineering patterns

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-drills-09-data-engineering.ipynb)

Companion to the [Python for data engineering](../learning/python-09-data-engineering.ipynb) chapter. The exercises build the small pieces of a pipeline: validation that keeps reasons, de-duplication, watermarks, partition paths, retry waits and reconciliation counts.

Try each exercise in the code cell below its question. Use the sample inputs to compare your result with the expected output. Expand **Show solution** when you want to review the approach.

## Exercise 1: validate a row and say why it fails

**Difficulty:** Medium · about 10 min

Write `validate_row(row)`. Return `(True, None)` for a valid row. Otherwise return `(False, reason)`, using exactly these reasons, checked in this order:

1. `"missing order_id"` when the `order_id` key is absent.
2. `"qty is not an integer"` when `qty` cannot be read as an `int`.
3. `"qty is negative"` when `qty` is below zero.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `validate_row({'order_id': 1, 'qty': '3'})` | `(True, None)` |
| `validate_row({'qty': '3'})` | `(False, 'missing order_id')` |
| `validate_row({'order_id': 1, 'qty': 'x'})` | `(False, 'qty is not an integer')` |

In [ ]:
def validate_row(row):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def validate_row(row):
    if "order_id" not in row:
        return False, "missing order_id"
    try:
        qty = int(row.get("qty", ""))
    except ValueError:
        return False, "qty is not an integer"
    if qty < 0:
        return False, "qty is negative"
    return True, None
```

</details>

## Exercise 2: keep the last version of each record

**Difficulty:** Medium · about 10 min

Write `dedupe_by_key(rows, key)`. When several rows share the same value in `key`, keep the **last** one. Return the kept rows in the order their keys first appeared.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `dedupe_by_key([{'id': 1, 'v': 'a'}, {'id': 2, 'v': 'b'}, {'id': 1, 'v': 'c'}], 'id')` | `[{'id': 1, 'v': 'c'}, {'id': 2, 'v': 'b'}]` |
| `dedupe_by_key([], 'id')` | `[]` |
| `dedupe_by_key([{'id': 1}], 'id')` | `[{'id': 1}]` |


*Hint: a dictionary keeps the first insertion position but takes the latest value.*

In [ ]:
def dedupe_by_key(rows, key):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def dedupe_by_key(rows, key):
    latest = {}
    for row in rows:
        latest[row[key]] = row
    return list(latest.values())
```

</details>

## Exercise 3: a watermark filter

**Difficulty:** Medium · about 10 min

Write `filter_newer(rows, watermark)`. Each row has a `ts` field holding an ISO date string. Keep the rows whose `ts` is **later** than `watermark`. Return a tuple `(kept_rows, new_watermark)`. `new_watermark` is the largest `ts` among the kept rows. If no row is kept, return the old watermark.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `filter_newer([{'ts': '2024-01-10'}, {'ts': '2024-01-12'}], '2024-01-10')` | `([{'ts': '2024-01-12'}], '2024-01-12')` |
| `filter_newer([{'ts': '2024-01-10'}], '2024-02-01')` | `([], '2024-02-01')` |
| `filter_newer([], '2024-01-01')` | `([], '2024-01-01')` |

In [ ]:
def filter_newer(rows, watermark):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def filter_newer(rows, watermark):
    kept = [row for row in rows if row["ts"] > watermark]
    if not kept:
        return kept, watermark
    return kept, max(row["ts"] for row in kept)
```

</details>

## Exercise 4: a Hive-style partition path

**Difficulty:** Medium · about 10 min

Write `partition_path(root, table, day)`. Return a `Path` for `<root>/<table>/dt=<day>/part-0000.jsonl`. Build it with `/`, not with string joins.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `partition_path('warehouse', 'orders', '2024-01-15').as_posix()` | `'warehouse/orders/dt=2024-01-15/part-0000.jsonl'` |
| `partition_path(Path('root'), 't', '2024-02-29').as_posix()` | `'root/t/dt=2024-02-29/part-0000.jsonl'` |
| `partition_path('/tmp', 't', '2024-01-01').is_absolute()` | `True` |

In [ ]:
from pathlib import Path

def partition_path(root, table, day):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
from pathlib import Path

def partition_path(root, table, day):
    return Path(root) / table / f"dt={day}" / "part-0000.jsonl"
```

</details>

## Exercise 5: exponential retry waits

**Difficulty:** Medium · about 10 min

Write `retry_delays(attempts, base_delay)`. Return the list of waits between attempts, with no jitter. There is one wait between each pair of attempts, so `attempts - 1` waits in all. Each wait is double the one before it, starting at `base_delay`.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `retry_delays(4, 0.5)` | `[0.5, 1.0, 2.0]` |
| `retry_delays(1, 1.0)` | `[]` |
| `retry_delays(0, 1)` | `[]` |

In [ ]:
def retry_delays(attempts, base_delay):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def retry_delays(attempts, base_delay):
    return [base_delay * 2 ** i for i in range(attempts - 1)]
```

</details>

## Exercise 6: count rejects by reason

**Difficulty:** Medium · about 10 min

Write `reason_counts(rejects)`. Each reject is a dictionary with a `reason` key. Return a plain `dict` that maps each reason to how many rejects had it.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `reason_counts([{'reason': 'a'}, {'reason': 'b'}, {'reason': 'a'}])` | `{'a': 2, 'b': 1}` |
| `reason_counts([])` | `{}` |
| `reason_counts([{'reason': 'bad'}])` | `{'bad': 1}` |


*Hint: `collections.Counter` counts for you. Convert it with `dict(...)` before you return it.*

In [ ]:
def reason_counts(rejects):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
from collections import Counter

def reason_counts(rejects):
    return dict(Counter(reject["reason"] for reject in rejects))
```

</details>

## Exercise 7: idempotent upsert with a content hash

**Difficulty:** Hard · about 20 min

Write `upsert_hashed(existing, batch)`. Rows have `id` (integer), `updated_at` (ISO UTC timestamp text), and a JSON-serializable `payload`. Existing rows also have `content_hash`. Return new rows sorted by id without mutating either input. Hash canonical JSON using SHA-256, sorted keys and separators `(',', ':')`. The greater `(updated_at, content_hash)` wins, so tied timestamps resolve independently of batch order. Identical versions do not change anything. A newer timestamp with unchanged content still advances version metadata, preventing an older changed payload from replacing it later.

Sample batch: `[{'id': 1, 'updated_at': '2026-01-01T00:00:00Z', 'payload': {'v': 2}}]`.
Expected result: one row with those values plus the SHA-256 content_hash of `{"v":2}`. Applying the batch again returns the same result.

In [ ]:
def upsert_hashed(existing, batch):
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import hashlib
import json

def upsert_hashed(existing, batch):
    # Rows use ISO UTC timestamps and JSON payloads. The hash is of canonical JSON.
    result = {r['id']: dict(r) for r in existing}
    for row in batch:
        digest = hashlib.sha256(json.dumps(row['payload'], sort_keys=True, separators=(',', ':')).encode()).hexdigest()
        candidate = {**row, 'content_hash': digest}
        previous = result.get(row['id'])
        if previous is None or (candidate['updated_at'], digest) > (previous['updated_at'], previous['content_hash']):
            result[row['id']] = candidate
    return [result[key] for key in sorted(result)]
```

</details>